This notebook is based on [this notebook](https://github.com/NielsRogge/Transformers-Tutorials/blob/master/TrOCR/Fine_tune_TrOCR_on_IAM_Handwriting_Database_using_Seq2SeqTrainer.ipynb) 

# Set-Up environment

## Packages

In [3]:
!pip install -q transformers


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [3]:
!pip install -q datasets jiwer


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [4]:
!pip install evaluate


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [4]:
!pip install dagshub


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [5]:
!pip install mlflow==2.21.2


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [6]:
!pip install mlflow_skinny==2.21.2


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [8]:
!pip install dotenv


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


## Set-Up directory

In [9]:
ls

chess-scoresheet-digitization-training/


In [10]:
import os

In [11]:
if "chess-scoresheet-digitization-training" not in os.listdir():
    !git clone https://github.com/benjaminkost/chess-scoresheet-digitization-training.git
    print("Git repository is cloned to current directory")
else:
    print("Git repository already in current directory")

Git repository already in current directory


In [7]:
cd chess-scoresheet-digitization-training/

/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/kaggle/working/chess-scoresheet-digitization-training


In [13]:
!git fetch

In [14]:
!git switch feature/16/create-trainings-pipeline-for-transformer-models

M	images_push_to_dockerhub.sh
Already on 'feature/16/create-trainings-pipeline-for-transformer-models'
Your branch is up to date with 'origin/feature/16/create-trainings-pipeline-for-transformer-models'.


In [15]:
!git pull

Already up to date.


## Imports

In [8]:
from transformers import TrOCRProcessor

In [9]:
import dagshub
import mlflow

In [10]:
from pathlib import Path

## Set-Up experiment tracking

In [11]:
dagshub.init(repo_owner="benjaminkost", repo_name="chess-scoresheet-digitization-training", mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=e14a1072-ea94-49cf-9647-0e17a412f688&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=51b7fb723eea9ee2a91dd3910e9b181e5956ef661d28f7a0bcc40e1237b5d58b




/usr/local/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Accessing as benjaminkost

Initialized MLflow to track repo "benjaminkost/chess-scoresheet-digitization-training"

Repository benjaminkost/chess-scoresheet-digitization-training initialized!

In [12]:
print(mlflow.get_tracking_uri())

https://dagshub.com/benjaminkost/chess-scoresheet-digitization-training.mlflow


# Set-up Trainer class

In [13]:
from transformers import VisionEncoderDecoderModel

In [ ]:
# Input data
## For Data
owner = "BenjaminKost"
dataset_name = "processed_hcs"
split = "train"
feature_column = "image"
target_column = "label"

## For Trainer, Processor and Model
processor_name = "trocr-base-handwritten-chess-notation-tokenizer"
processor = TrOCRProcessor.from_pretrained(f"{owner}/{processor_name}")
model_name = "TrOCR-Base-Fined-On-HCS" # model = VisionEncoderDecoderModel.from_pretrained("./models/checkpoint-4119")

run_name = "Model-TrOCR-Base-Fined-On-HCS-Training-Nr_1"
experiment_name = "Train for model digitalizing hand written chess game notations"
model_flavor = "pytorch"

In [ ]:
os.environ["HF_MODEL_URI"]="BenjaminKost/trocr-base-handwritten-chess-notation"
os.environ["DAGSHUB_MLFLOW_TRACKING_USERNAME"]="benjaminkost"
os.environ["DAGSHUB_REPOSITORY"]="chess-scoresheet-digitization-training"

# Train model

In [ ]:
from src.scripts.pipelines.training_pipeline import training_pipeline_without_preprocessing_for_transformer_models

In [ ]:
ls

In [14]:
dir_path = Path.cwd()
output_dir = str(dir_path / "models")

In [ ]:
# Call pipeline
training_pipeline_without_preprocessing_for_transformer_models(
    owner=owner,
    dataset_name=dataset_name,
    split=split,
    feature_column=feature_column,
    target_column=target_column,
    processor=processor,
    model_name=model_name,
    run_name=run_name,
    experiment_name=experiment_name,
    model_flavor=model_flavor,
    predict_with_generate=True,
    eval_strategy="steps",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=True,
    output_dir=output_dir,
    logging_steps=2,
    save_steps=2000,
    eval_steps=500,
    disable_tqdm=False,
    report_to="none",
    max_length=64,
    early_stopping=True,
    no_repeat_ngram_size=3,
    length_penalty=2.0,
    num_beams=4,
)

2025-08-20 17:56:37,251 - src.scripts.scripts_for_steps.ingest_data_strategy - INFO - Dataset 'BenjaminKost/processed_hcs' loaded successfully!
2025-08-20 17:56:37,251 - src.scripts.scripts_for_steps.ingest_data_strategy - INFO - Dataset 'BenjaminKost/processed_hcs' loaded successfully!
2025-08-20 17:56:37,252 - src.scripts.pipelines.training_pipeline - INFO - Test if module was newly imported successfully
2025-08-20 17:56:55,157 - src.scripts.scripts_for_steps.data_splitter_strategy - INFO - Train-Test-Split done successfully: Training data: 10984, Test data: 2747
2025-08-20 17:56:55,158 - src.scripts.steps.data_encoding - INFO - Dataset was encoded
2025-08-20 17:56:55,159 - src.scripts.steps.data_encoding - INFO - Dataset was encoded
2025-08-20 17:56:55,201 - src.scripts.steps.train_model - INFO - Training step started
2025-08-20 17:56:55,202 - src.scripts.scripts_for_steps.trainer_module - INFO - MLflow uri is: https://dagshub.com/benjaminkost/chess-scoresheet-digitization-training.

Step,Training Loss,Validation Loss,Cer
500,0.268300,7.738008,16.058213
1000,0.348800,6.072527,15.183586
1500,0.263300,6.660601,11.711678
2000,0.183300,6.977143,15.921389
2500,0.248700,7.915234,18.281284
3000,0.444800,7.690702,18.058213


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 64, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


🏃 View run Model-TrOCR-Base-Fined-On-HCS-Training-Nr_1 at: https://dagshub.com/benjaminkost/chess-scoresheet-digitization-training.mlflow/#/experiments/3/runs/eaf4679242b04b71ad5c6d0ac184b5a0
🧪 View experiment at: https://dagshub.com/benjaminkost/chess-scoresheet-digitization-training.mlflow/#/experiments/3


# Track afterwards

In [17]:
run_name = "Model-TrOCR-Base-Fined-On-HCS-Training-Nr_1"
experiment_name = "Train for model digitalizing hand written chess game notations"
model_flavor = "pytorch"

model = VisionEncoderDecoderModel.from_pretrained("./models/checkpoint-4119")

mlflow.set_experiment(experiment_name)

# Log parameters, metrics, model signature
mlflow.autolog()

with mlflow.start_run(run_name=run_name):

    #unwrapped_model = self.get_trainer().accelerator.unwrap_model(model)
    model_metadata = mlflow.pytorch.log_model(model, artifact_path="model")

2025/08/21 08:47:50 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2025/08/21 08:47:50 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.
2025/08/21 08:47:50 WARNING mlflow.utils.autologging_utils: MLflow keras autologging is known to be compatible with 3.0.2 <= keras <= 3.8.0, but the installed version is 3.10.0. If you encounter errors during autologging, try upgrading / downgrading keras to a compatible version, or try upgrading MLflow.
2025/08/21 08:47:50 INFO mlflow.tracking.fluent: Autologging successfully enabled for keras.
2025/08/21 08:47:50 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 0.24.1 <= scikit-learn <= 1.6.1, but the installed version is 1.7.0. If you encounter errors during autologging, try upgrading / downgrading

🏃 View run Model-TrOCR-Base-Fined-On-HCS-Training-Nr_1 at: https://dagshub.com/benjaminkost/chess-scoresheet-digitization-training.mlflow/#/experiments/3/runs/bc9dafab08414730bd957df4df513b50
🧪 View experiment at: https://dagshub.com/benjaminkost/chess-scoresheet-digitization-training.mlflow/#/experiments/3


# Upload to Huggingface

Note that after training, you can easily load the model using the .from_pretrained(output_dir) method.

For inference on new images, I refer to my inference notebook, that can also be found in my Transformers Tutorials repository on Github.

In [ ]:
model_name="trocr-base-handwritten-chess-notation"

In [ ]:
model = VisionEncoderDecoderModel.from_pretrained(f"{output_dir}/checkpoint-4119")

In [ ]:
tokenizer = TrOCRProcessor.from_pretrained(f"{output_dir}/checkpoint-4119")

In [ ]:
# token does not work anymore -> new one has to be 
# here should be a huggingface token

In [ ]:
# here push model with .push_to_hub()

In [ ]:
# here push tokenizer with .push_to_hub()

# Export kaggle directory

In [ ]:
ls

In [23]:
!zip -r trocr-checkpoint-4119.zip /kaggle/working/chess-scoresheet-digitization-training/models/checkpoint-4119

  adding: kaggle/working/chess-scoresheet-digitization-training/models/checkpoint-4119/ (stored 0%)
  adding: kaggle/working/chess-scoresheet-digitization-training/models/checkpoint-4119/scheduler.pt (deflated 55%)
  adding: kaggle/working/chess-scoresheet-digitization-training/models/checkpoint-4119/generation_config.json (deflated 40%)
  adding: kaggle/working/chess-scoresheet-digitization-training/models/checkpoint-4119/trainer_state.json (deflated 81%)
  adding: kaggle/working/chess-scoresheet-digitization-training/models/checkpoint-4119/preprocessor_config.json (deflated 48%)
  adding: kaggle/working/chess-scoresheet-digitization-training/models/checkpoint-4119/merges.txt (deflated 53%)
  adding: kaggle/working/chess-scoresheet-digitization-training/models/checkpoint-4119/vocab.json (deflated 59%)
  adding: kaggle/working/chess-scoresheet-digitization-training/models/checkpoint-4119/tokenizer.json (deflated 82%)
  adding: kaggle/working/chess-scoresheet-digitization-training/model

In [24]:
from IPython.display import FileLink

FileLink(r'../../../Downloads/trocr-checkpoint-4119.zip')

/kaggle/working/trocr-checkpoint-4119.zip